# Liputan 6 SMATCH adjacency demo

Liputan 6 stores one AMR per `{doc_id}_{sent_idx}.txt` inside a ZIP. This notebook groups the sentence graphs by document, sorts them by sentence index, and demonstrates the pairwise SMATCH matrix used by the production script.

In [ ]:
import re, zipfile
from collections import defaultdict
from pathlib import Path, PurePosixPath

import numpy as np
import penman
import smatch

In [ ]:
ZIP_PATH = Path('data/liputan6/amr_graphs/archives/train.zip')
DOC_ID = None  # set e.g. '290735', or leave None to choose the first multi-sentence document
FILE_RE = re.compile(r'^(?P<doc_id>.+)_(?P<sent_idx>\d+)\.txt$')

In [ ]:
def graph_to_oneline(graph):
    formatted = penman.format(penman.configure(graph))
    return ' '.join(line.strip() for line in formatted.splitlines()
                    if line.strip() and not line.lstrip().startswith('#'))

def smatch_f_score(first, second):
    try:
        best, first_n, second_n = smatch.get_amr_match(first, second, sent_num=1)
        return smatch.compute_f(best, first_n, second_n)[2]
    finally:
        smatch.match_triple_dict.clear()

def adjacency(amrs):
    matrix = np.zeros((len(amrs), len(amrs)), dtype=np.float32)
    for i in range(len(amrs)):
        for j in range(i + 1, len(amrs)):
            matrix[i, j] = matrix[j, i] = smatch_f_score(amrs[i], amrs[j])
    return matrix

In [ ]:
with zipfile.ZipFile(ZIP_PATH) as archive:
    grouped = defaultdict(list)
    for info in archive.infolist():
        match = FILE_RE.fullmatch(PurePosixPath(info.filename).name)
        if match:
            grouped[match.group('doc_id')].append((int(match.group('sent_idx')), info.filename))

    selected_doc = DOC_ID or next(doc for doc, files in grouped.items() if len(files) > 1)
    members = sorted(grouped[selected_doc])
    sentence_indices = [idx for idx, _ in members]
    amrs = []
    for _, member in members:
        graphs = penman.loads(archive.read(member).decode('utf-8-sig'))
        assert len(graphs) == 1, f'{member}: expected one graph'
        amrs.append(graph_to_oneline(graphs[0]))

matrix = adjacency(amrs)
print(f'document={selected_doc}, sentence_indices={sentence_indices}')
print(f'shape={matrix.shape}, symmetric={np.allclose(matrix, matrix.T)}')
matrix